In [ ]:
# Install required Python packages (run once per environment):
# - langchain / langchain-experimental: core LangChain framework + experimental graph transformer
# - langchain-openai: OpenAI chat model integration (used to call GPT-4o)
# - python-dotenv: load environment variables (e.g. API keys) from a local .env file
# - pyvis: render interactive network graphs to an HTML file
%pip install --upgrade langchain langchain-experimental langchain-openai python-dotenv pyvis

In [ ]:
# Load secrets (API keys etc.) from a .env file into process environment variables,
# so that we never hard-code credentials in the notebook.
from dotenv import load_dotenv
import os

# Read key=value lines from `.env` in the current working directory
# and inject them into os.environ.
load_dotenv()

# Pull the OpenAI API key out of the environment.
# The .env file must contain a line like:  OPENAI_API_KEY=sk-...
# LangChain's ChatOpenAI will automatically pick up OPENAI_API_KEY from env,
# so storing it here is mainly for explicit access if needed.
api_key = os.getenv("OPENAI_API_KEY")

### LLM Graph Transformer
Using GPT-4o in all examples.

In [ ]:
# Build the LLM-powered graph extraction pipeline.
# LLMGraphTransformer wraps an LLM and turns raw text Documents
# into structured (Node, Relationship) triples — i.e. a knowledge graph.
from langchain_experimental.graph_transformers import LLMGraphTransformer
from langchain_core.documents import Document
from langchain_openai import ChatOpenAI

# temperature=0 -> deterministic output (important for reproducible graph extraction).
# model_name="gpt-4o" -> use OpenAI's GPT-4o, which is strong at structured extraction.
llm = ChatOpenAI(temperature=0, model_name="gpt-4o")

# Default transformer: no schema constraints, the LLM decides node/relationship types freely.
graph_transformer = LLMGraphTransformer(llm=llm)

### Extract graph data

In [ ]:
# Source text we want to convert into a knowledge graph.
# Here: an English Wikipedia-style description of Kyushu University.
# The LLM will read this paragraph and extract entities (people, places,
# organizations, etc.) along with the relationships between them.
text = """
Kyushu University (九州大学, Kyūshū daigaku), abbreviated to Kyudai (九大, Kyūdai), is a public research university located in Fukuoka, Japan, on the island of Kyushu. It was the fourth Imperial University in Japan, ranked as one of the top ten Designated National University and selected as a Top Type university of Top Global University Project by the Japanese government. It is considered one of the most prestigious universities in Japan.

The university traces its roots back to the Medical Training Center established in 1867 (the year of the Meiji Restoration) by Fukuoka Domain. After several iterations, it was established as Fukuoka Medical College of Kyoto Imperial University in 1903, and became independent as Kyushu Imperial University in 1911. After World War II in 1947, Kyushu Imperial University was renamed Kyushu University. In 2003, Kyushu University merged with Kyushu Institute of Design to form a new Kyushu University. As of May 2017, the university had 11 undergraduate schools and 18 graduate schools, with a total of approximately 18,800 students.

Kyushu University has produced many notable alumni, including the Prime Minister of Japan Hirobumi Ito, Nobel Prize laureate in Physics Leo Esaki, and prominent scientists in various fields. The university maintains active international collaborations and is recognized for its strong research programs in engineering, medicine, and the humanities. Its main campus is the Ito Campus, located in Nishi-ku, Fukuoka, which is one of the largest university campuses in Japan.
"""

In [ ]:
# Wrap the raw text into a list of LangChain Document objects
# (LLMGraphTransformer operates on Documents, not plain strings).
documents = [Document(page_content=text)]

# Asynchronously send the document(s) to the LLM and ask it to extract
# nodes + relationships. The result is a list of GraphDocument objects,
# one per input Document. We `await` because aconvert_to_graph_documents
# is a coroutine — Jupyter supports top-level await natively.
graph_documents = await graph_transformer.aconvert_to_graph_documents(documents)

In [ ]:
# Inspect the extraction result of the first (and only) document:
# - .nodes:         list of Node(id, type, properties)
# - .relationships: list of Relationship(source, target, type, properties)
print(f"Nodes:{graph_documents[0].nodes}")
print(f"Relationships:{graph_documents[0].relationships}")

#### Visualize graph

In [ ]:
from pyvis.network import Network

def visualize_graph(graph_documents):
    """Render the first GraphDocument as an interactive HTML network graph."""

    # Create an empty pyvis Network canvas.
    # - directed=True: edges are arrows (relationships have a direction)
    # - bgcolor / font_color: dark theme
    net = Network(height="1200px", width="100%", directed=True,
                      notebook=False, bgcolor="#222222", font_color="white")

    # Pull nodes and relationships from the first GraphDocument.
    nodes = graph_documents[0].nodes
    relationships = graph_documents[0].relationships

    # Build an id -> Node lookup so we can validate edges quickly.
    node_dict = {node.id: node for node in nodes}

    # Keep only edges where BOTH endpoints are present in node_dict.
    # The LLM can occasionally reference a target node that wasn't formally
    # listed in `nodes`, so we defensively filter those out.
    valid_edges = []
    valid_node_ids = set()
    for rel in relationships:
        if rel.source.id in node_dict and rel.target.id in node_dict:
            valid_edges.append(rel)
            valid_node_ids.update([rel.source.id, rel.target.id])

    # (Diagnostic) Collect every node id that appears in any relationship,
    # regardless of validity. Currently unused, but useful when debugging
    # orphan nodes.
    connected_node_ids = set()
    for rel in relationships:
        connected_node_ids.add(rel.source.id)
        connected_node_ids.add(rel.target.id)

    # Add nodes to the pyvis network.
    # `group=node.type` makes pyvis color-code nodes by their type
    # (Person / Organization / Location / ...).
    for node_id in valid_node_ids:
        node = node_dict[node_id]
        try:
            net.add_node(node.id, label=node.id, title=node.type, group=node.type)
        except:
            continue  # ignore duplicate-id / malformed-id errors

    # Add edges to the pyvis network. The edge label is the relationship type,
    # lower-cased for readability (e.g. "located_in" instead of "LOCATED_IN").
    for rel in valid_edges:
        try:
            net.add_edge(rel.source.id, rel.target.id, label=rel.type.lower())
        except:
            continue  # ignore edges that reference unknown nodes

    # Configure the force-directed physics simulation that lays out the graph.
    # forceAtlas2Based gives a clean, well-separated layout for small/medium graphs.
    net.set_options("""
            {
                "physics": {
                    "forceAtlas2Based": {
                        "gravitationalConstant": -100,
                        "centralGravity": 0.01,
                        "springLength": 200,
                        "springConstant": 0.08
                    },
                    "minVelocity": 0.75,
                    "solver": "forceAtlas2Based"
                }
            }
            """)

    # Write the interactive graph to a standalone HTML file in the project dir.
    output_file = "knowledge_graph.html"
    net.save_graph(output_file)
    print(f"Graph saved to {os.path.abspath(output_file)}")

    # Best-effort: open the HTML file in the default web browser.
    try:
        import webbrowser
        webbrowser.open(f"file://{os.path.abspath(output_file)}")
    except:
        print("Could not open browser automatically")

# Render and open the graph extracted from the Kyushu University text.
visualize_graph(graph_documents)

### Extract specific types of nodes

In [ ]:
# Constrain the schema: only let the LLM emit nodes whose type is one of these.
# This produces a cleaner, more focused graph and prevents the LLM from
# inventing dozens of ad-hoc node types.
allowed_nodes = ["Person", "Organization", "Location", "Award", "ResearchField"]

# Re-build the transformer with the node-type whitelist.
graph_transformer_nodes_defined = LLMGraphTransformer(llm=llm, allowed_nodes=allowed_nodes)

# Run extraction again on the same documents with the new constraints.
graph_documents_nodes_defined = await graph_transformer_nodes_defined.aconvert_to_graph_documents(documents)

In [ ]:
# Print the schema-constrained extraction so we can compare it
# against the unconstrained run above.
print(f"Nodes:{graph_documents_nodes_defined[0].nodes}")
print(f"Relationships:{graph_documents_nodes_defined[0].relationships}")

### Extract specific types of relationships

In [ ]:
# Tighten the schema even further: also pin the *relationship* schema.
# Each tuple is (source_node_type, relationship_type, target_node_type).
# The LLM is now forbidden from inventing new edge types or applying
# an edge between unexpected node-type pairs.
allowed_nodes = ["Person", "Organization", "Location", "Award", "ResearchField"]
allowed_relationships = [
    ("Organization", "LOCATED_IN", "Location"),
    ("Organization", "MERGED_WITH", "Organization"),
    ("Organization", "RENAMED_TO", "Organization"),
    ("Person", "ALUMNUS_OF", "Organization"),
    ("Person", "RECEIVED", "Award"),
    ("Organization", "RESEARCH_IN", "ResearchField")
]

# Build a transformer with both node and relationship whitelists.
graph_transformer_rel_defined = LLMGraphTransformer(
  llm=llm,
  allowed_nodes=allowed_nodes,
  allowed_relationships=allowed_relationships
)

# Run extraction one more time with the strictest schema.
graph_documents_rel_defined = await graph_transformer_rel_defined.aconvert_to_graph_documents(documents)

In [ ]:
# Render the strictly-typed graph using the same pyvis helper as before.
# This will overwrite knowledge_graph.html with the constrained graph.
visualize_graph(graph_documents_rel_defined)